<a href="https://colab.research.google.com/github/astronerd21/Deep-Learning-Oil-Slick-Detection/blob/main/Lookalike_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# False Positive / Lookalike Analysis
**Goal:** Inspect "NO OIL" (0) samples that the model incorrectly predicted as "OIL" (1). This notebook performs an automated evaluation to identify false positives and visualizes their radar signatures.

**Methodology:** Analysis of false positive IDs exported from the model evaluation pipeline, followed by a comparative visualization of VV and VH SAR channel signatures and their pixel intensity distributions (histograms).

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

%pip install rasterio matplotlib numpy

os.makedirs('/content/data', exist_ok=True)

if not os.path.exists('/content/data/images_s1'):
    print("Extracting satellite images from Drive...")
    !tar -xf /content/drive/MyDrive/OilSlickProject/data/OilSlick/OilSlick-images_s1-00.tar -C /content/data/
    !tar -xf /content/drive/MyDrive/OilSlickProject/data/OilSlick/OilSlick-images_s1-01.tar -C /content/data/
    print("Extraction complete!")
else:
    print("Images are already extracted.")

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Paths to the finished evaluations
fp_paths = {
    "Baseline Random": Path("/content/drive/MyDrive/OilSlickProject/data/OilSlick/results/baseline_random/evaluation_outputs/false_positives_test_clean.txt"),
    "ResNet Geo OOD": Path("/content/drive/MyDrive/OilSlickProject/data/OilSlick/results/geographic_ood/evaluation_outputs/false_positives_geo_test_clean.txt"),
    "TerraMind Geo OOD": Path("/content/drive/MyDrive/OilSlickProject/data/OilSlick/results/terramind_ood/evaluation_outputs/false_positives_geo_test_clean.txt")
}

# Function to read the lists
def load_fp_list(path):
    if not path.exists():
        print(f"WARNING: File not found -> {path}")
        return []
    with open(path, "r") as f:
        return [line.strip() for line in f if line.strip()]

# Load lists into a dictionary
fps = {name: load_fp_list(path) for name, path in fp_paths.items()}

for name, ids in fps.items():
    print(f"{name}: {len(ids)} False Positives loaded.")

# Helper functions for plotting
data_dir = Path("/content/data/images_s1/")

def load_and_normalize_sar(filepath: Path) -> np.ndarray:
    with rasterio.open(filepath) as src:
        data = src.read().astype(np.float32)
    for c in range(data.shape[0]):
        channel_mean = np.mean(data[c])
        channel_std = np.std(data[c])
        if channel_std > 0:
            data[c] = (data[c] - channel_mean) / channel_std
    return data

def plot_top_5(model_name):
    ids = fps.get(model_name, [])[:5]  # Hier wird auf 5 abgeschnitten
    if not ids:
        return
    
    num_samples = len(ids)
    fig, axes = plt.subplots(num_samples, 4, figsize=(20, 4 * num_samples))
    fig.suptitle(f"Lookalike Analysis: {model_name} (Top {num_samples})", fontsize=16, y=1.02)

    if num_samples == 1:
        axes = np.array([axes])

    for idx, filename in enumerate(ids):
        # Fallback in case the .txt files are missing the .tif extension
        img_file = filename if filename.endswith('.tif') else f"{filename}_s1.tif"
        filepath = data_dir / img_file
        
        try:
            data = load_and_normalize_sar(filepath)
            axes[idx, 0].imshow(data[0], cmap='gray'); axes[idx, 0].set_title(f"VV | {filename}"); axes[idx, 0].axis('off')
            axes[idx, 1].hist(data[0].ravel(), bins=50, color='blue', alpha=0.7); axes[idx, 1].set_title("VV Hist")
            axes[idx, 2].imshow(data[1], cmap='gray'); axes[idx, 2].set_title(f"VH | {filename}"); axes[idx, 2].axis('off')
            axes[idx, 3].hist(data[1].ravel(), bins=50, color='orange', alpha=0.7); axes[idx, 3].set_title("VH Hist")
        except Exception as e:
            axes[idx, 0].set_title(f"Error at {filename}"); axes[idx, 0].axis('off')
            print(f"Error loading {filename}: {e}")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_top_5("Baseline Random")

**Physical Classification (Baseline Random):**

* **Dominant Phenomenon:** The model is entirely dependent on pixel intensity and struggles with any form of signal dampening, lacking spatial awareness. It falls for a wide variety of standard lookalikes.
* **Specific Observations:** * **Low-Wind Areas:** Clear failures on calm water where the signal drops completely (e.g., `neg_00357`, `neg_00857`), characterized by extremely narrow, spiked histograms.
    * **Biogenic Films:** Image `neg_00640` shows the classic swirling eddies of natural organic films.
    * **Ship Wakes:** Image `neg_00342` features the distinct, linear dark streaks caused by a ship's propeller turbulence.
    * **Artifacts:** Image `neg_01166` contains bright interference artifacts, causing the model to misclassify the surrounding dark background.
* **Model Behavior:** The baseline has not learned abstract geometric features. It classifies anything with a significantly low VV-channel backscatter as oil, regardless of whether it is a wake, a swirl, or just a windless patch of ocean.

In [ ]:
plot_top_5("ResNet Geo OOD")

**Physical Classification (ResNet Geo OOD):**

* **Dominant Phenomenon:** Structural lookalikes and data processing artifacts. The model has learned to recognize shapes and edges but lacks the broader geographic context to dismiss them.
* **Specific Observations:**
    * **Complex Structures:** It correctly ignores simple flat, calm waters (unlike the baseline) but gets tricked by highly structured biogenic streaks (`neg_00113`) and ship wakes (`neg_00342`).
    * **Land/Sea Boundaries:** Images like `neg_00254` and `neg_00311` show the model struggling with coastal wind shadows or the edges of landmasses/sea ice.
    * **NoData Artifacts:** A glaring failure is `neg_04470`. The completely black image and the perfectly flat spike in the histogram at 0 indicate a missing data wedge or processing artifact, which the ResNet falsely interpreted as a massive, uniform oil slick.
* **Model Behavior:** The ResNet is a massive step up from the baseline, utilizing spatial convolutions to find "slick-like" shapes. However, its limited receptive field means it cannot differentiate between the structural edge of an algal bloom and an actual oil spill.

In [ ]:
plot_top_5("TerraMind Geo OOD")

**Physical Classification (TerraMind Geo OOD):**

* **Dominant Phenomenon:** Morphologically complex, "true" physical lookalikes. The Foundation Model only fails when the natural phenomenon perfectly mimics the physical and structural signature of an oil spill.
* **Specific Observations:**
    * **Overlap with ResNet:** There is a high overlap with the ResNet model (e.g., `neg_00113`, `neg_00342`, `neg_00254`). Both models struggle with biogenic streaks and ship wakes because they look identical to oil in SAR data.
    * **Extreme Complexity:** Image `neg_00617` shows an incredibly complex pattern (likely a massive algal bloom, biogenic film, or fractured sea ice).
    * **Artifact Robustness:** Notably, the pure NoData artifact (`neg_04470`) that fooled the ResNet is absent here. TerraMind successfully ignored it.
* **Model Behavior:** The Foundation Model demonstrates a robust understanding of standard SAR physics. It successfully filters out sensor artifacts and simple low-wind areas. Its false positives are strictly limited to phenomena that dampen capillary waves in the exact same morphological patterns as mineral oil, proving that single-modality SAR data has a hard physical limit without external context (like AIS or optical data).

In [ ]:
# Convert lists to sets to find the intersection
sets = {name: set(ids) for name, ids in fps.items()}

# Intersection: Images that were a False Positive in ALL THREE models
uncommon_lookalikes = sets["Baseline Random"] & sets["ResNet Geo OOD"] & sets["TerraMind Geo OOD"]

print("=== OVERLAP ANALYSIS ===")
print(f"Number of 'impossible lookalikes' (False alarm in all 3 models): {len(uncommon_lookalikes)}\n")

print("Top 10 IDs of these hard cases:")
for img_id in list(uncommon_lookalikes)[:10]:
    print(f" - {img_id}")

# visualize the hardest case
if uncommon_lookalikes:
    hard_case = list(uncommon_lookalikes)[0]
    print(f"\nVisualizing an exemplary hard case: {hard_case}")
    fps["Hardcase Demo"] = [hard_case]
    plot_top_5("Hardcase Demo")

Ich fände für folgende bilder Grad Cam am ende hier interresant:
- neg_01319_s1.tif
- neg_00212_s1.tif
- neg_00311_s1.tif


Außerdem würde ich vlt. die anzahl der angezeigten bilder auf 5/6 begrenzen bei den einzelnen modellen.

Die beschreinbung muss angepasst werden, da es neue bilder sind.